> ⚠️ **机器翻译提示**：本notebook的文字说明由机器翻译生成，尚未经过母语者校对，请谨慎参考。代码与数字结果与越南语原版notebook完全一致。

# 第10单元 — 因子分析、聚类分析与SEM/多层次模型入门
第10单元专用notebook — 在班级数据上重现EFA（2因子），并在真实的195所学校PISA数据上进行K均值聚类和轮廓系数分析。需要安装 `factor_analyzer` 和 `scikit-learn`（Colab已预装）。

In [1]:
# 请先运行此单元格（Colab：点击 ▶）。无需额外安装。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240名学生的班级" —— 与第03-08单元使用的
    同一个seed=2026，与代码库中的lop_hoc_240.csv/.sav完全一致
    （非真实数据，但可100%复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式学习"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式学习")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式学习") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df:", df.shape)

已生成 df: (240, 14)


## 1. 探索性因子分析（EFA）
🎯 **该方法回答什么问题？** EFA回答：6个可观测项目（h1-h3, a1-a3）能否归纳为少数几个潜在因子，以及每个项目"属于"哪个潜在因子？

In [2]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
print("n =", len(items))

# 手动计算 Bartlett 与 KMO（避免 factor_analyzer 在新版 numpy 下的兼容性错误）
R = items.corr().values.copy()
n_obs, p_vars = items.shape
det_R = np.linalg.det(R)
chi2 = -(n_obs - 1 - (2*p_vars + 5)/6) * np.log(det_R)
dof = p_vars*(p_vars-1)/2
p_val = 1 - stats.chi2.cdf(chi2, dof)

R_inv = np.linalg.inv(R)
D = np.diag(1/np.sqrt(np.diag(R_inv)))
partial_corr = -D @ R_inv @ D
np.fill_diagonal(partial_corr, 0)
R2 = R.copy(); np.fill_diagonal(R2, 0)
kmo_model = np.sum(R2**2) / (np.sum(R2**2) + np.sum(partial_corr**2))

print(f"Bartlett: chi2={chi2:.2f}, df={dof:.0f}, p={p_val:.2e}")
print(f"KMO = {kmo_model:.3f}")

n = 237
Bartlett: chi2=313.28, df=15, p=0.00e+00
KMO = 0.688


#### 📤 真实输出
KMO=0.688（可接受水平，>0.6）。Bartlett p<0.001（显著）→ 数据满足进行EFA的条件。

In [3]:
from factor_analyzer import FactorAnalyzer
import pandas as pd

fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
loadings = pd.DataFrame(fa.loadings_, index=items.columns, columns=["因子1", "因子2"])
print(loadings.round(3))

ev, _ = fa.get_eigenvalues()
print("\n特征值：", ev[:4].round(3))

      因子1    因子2
h1 -0.043  0.747
h2 -0.003  0.603
h3 -0.096  0.670
a1  0.725 -0.092
a2  0.722 -0.083
a3  0.695 -0.016

特征值： [2.209 1.732 0.605 0.521]


#### 📤 真实输出
a1-a3 明显载荷在因子1上（0.70-0.73）；h1-h3 明显载荷在因子2上（0.60-0.75）。特征值：因子1=2.21，因子2=1.73（均>1）。✅ EFA "重新发现"了用于生成该数据的2个潜在概念（兴趣、焦虑）——确认该方法运作正确。

## 2. K均值聚类分析 —— 真实PISA数据（195所学校）
🎯 **该方法回答什么问题？** K均值回答：195所学校能否根据5个WLE指标分组为若干个具有相似"画像"（profile）的聚类——与EFA不同的是，这里分组的对象是"学校"，而不是"变量"。

In [4]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
print("n =", len(d))

for k in [2,3,4]:
    km = KMeans(n_clusters=k, random_state=2026, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette={sil:.3f}, 聚类大小={list(np.bincount(km.labels_))}")

n = 195
k=2: silhouette=0.252, 聚类大小=[np.int64(63), np.int64(132)]
k=3: silhouette=0.180, 聚类大小=[np.int64(64), np.int64(49), np.int64(82)]
k=4: silhouette=0.233, 聚类大小=[np.int64(25), np.int64(52), np.int64(102), np.int64(16)]


#### 📤 真实输出
k=2 的轮廓系数最高（0.252）——纯统计意义上的最优。k=3（0.180）的轮廓系数较低，但解释性更强——这是统计最优与可解释性之间的经典权衡。

In [5]:
km3 = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
d2 = d.copy(); d2["cluster"] = km3.labels_
print(d2.groupby("cluster")[vars_].mean().round(3))
print("\n各聚类的大小：")
print(d2["cluster"].value_counts().sort_index())

         EDULEAD  NEGSCLIM  STAFFSHORT  EDUSHORT  DIGPREP
cluster                                                  
0          1.422    -0.621      -0.620    -0.675    1.216
1          0.691     0.703       1.334     1.514   -0.259
2          0.297    -0.441      -0.095    -0.194   -0.233

各聚类的大小：
cluster
0    64
1    49
2    82
Name: count, dtype: int64


#### 📤 真实输出
聚类1（n=49）的 STAFFSHORT=+1.33，EDUSHORT=+1.51（均为高正值）→ "多重困境"画像：同时存在师资短缺和教学资源短缺。聚类0（n=64）的 EDULEAD=+1.42，DIGPREP=+1.22 → "领导力强、数字化准备度高"画像。

### ⚠️ K均值对初始点敏感 —— PSPP给出与Python不同的聚类大小
在PSPP中用同样的数据、同样的k=3运行 `QUICK CLUSTER`，得到的聚类大小为37/128/30 —— 与Python的64/49/82明显不同。这并非错误，而是K均值算法已知的特性：该算法对初始质心（initial centroids）非常敏感，而两款软件使用了不同的初始化策略。

In [6]:
# 演示：用不同的随机初始化多次运行K均值（不固定random_state）
results = []
for trial in range(5):
    km_trial = KMeans(n_clusters=3, n_init=1, random_state=trial).fit(X)
    results.append(sorted(np.bincount(km_trial.labels_)))
print("5次运行中各次的聚类大小（已排序），每次仅用n_init=1（单次初始化）：")
for r in results:
    print(" ", r)
print("\n-> 当只用单次初始化（n_init=1）时，结果在各次运行之间会有波动。")
print("使用n_init=10（推荐的默认值）可在10次尝试中取得最稳定的结果。")

5次运行中各次的聚类大小（已排序），每次仅用n_init=1（单次初始化）：
  [np.int64(45), np.int64(66), np.int64(84)]
  [np.int64(29), np.int64(56), np.int64(110)]
  [np.int64(40), np.int64(61), np.int64(94)]
  [np.int64(50), np.int64(66), np.int64(79)]
  [np.int64(34), np.int64(71), np.int64(90)]

-> 当只用单次初始化（n_init=1）时，结果在各次运行之间会有波动。
使用n_init=10（推荐的默认值）可在10次尝试中取得最稳定的结果。


## 3. 轮廓系数（Silhouette）—— 公式与直观理解
🎯 **该方法回答什么问题？** 轮廓系数衡量每个数据点的聚类质量：该点是否"紧密"地位于自己所属的聚类内（离同聚类的点近、离其他聚类的点远）？

In [7]:
from sklearn.metrics import silhouette_samples
sil_vals = silhouette_samples(X, km3.labels_)
d3 = d.copy(); d3["cluster"] = km3.labels_; d3["silhouette"] = sil_vals
print("各聚类的平均轮廓系数：")
print(d3.groupby("cluster")["silhouette"].mean().round(3))
print(f"\n轮廓系数为负的学校数量（可能被错误分配到聚类）：{(sil_vals<0).sum()} / {len(sil_vals)}")

各聚类的平均轮廓系数：
cluster
0    0.078
1    0.110
2    0.301
Name: silhouette, dtype: float64

轮廓系数为负的学校数量（可能被错误分配到聚类）：34 / 195


#### 📤 真实输出
部分学校的轮廓系数为负 —— 表明聚类之间的边界并不完全清晰（与总体轮廓系数仅为0.18的水平一致，并不高）。这正是必须如实报告聚类质量的重要原因，而不应将结果呈现为聚类之间完全分离。